# Switching providers: Gemini ↔ OpenAI (same code)

Vendor lock-in is a production risk. If your feature code calls the SDK
directly, swapping Gemini for OpenAI (or adding Claude later) means rewriting
every call site.

genai-prod-kit avoids this by putting one rule between your code and the model:
a provider only has to expose `name` and `generate(prompt, *, model) -> LLMResult`.
The Gateway, pricing, eval, PII, and drift layers never import a vendor SDK.

This notebook calls **the exact same Gateway code through two providers** and
shows that only the injected object changes.

In [ ]:
import os, sys
sys.path.insert(0, "../src")

gemini_key = os.environ["GEMINI_API_KEY"]
openai_key = os.environ["OPENAI_API_KEY"]

## 1. Two providers, one interface

Both classes satisfy the same `LLMProvider` protocol. Nothing else in the kit
needs to know which one it received.

In [ ]:
from genai_prod_kit.providers.gemini import GeminiProvider
from genai_prod_kit.providers.openai import OpenAIProvider

gemini = GeminiProvider(gemini_key)
openai = OpenAIProvider(openai_key)

print(gemini.name, "->", type(gemini).__name__)
print(openai.name, "->", type(openai).__name__)

## 2. The call site does not change

`run_demo` takes a provider and a model. It is identical for every vendor —
the only per-vendor inputs are the injected provider and the model string.
We use the cheapest model on each side (`gemini-2.5-flash`, `gpt-5.4-nano`).

In [ ]:
from genai_prod_kit.gateway import call_llm
from genai_prod_kit.sinks.jsonl import JsonlSink

sink = JsonlSink("nb_switch_invocations.jsonl")

def run_demo(provider, model):
    res = call_llm(
        "What is the capital of Japan? Answer in one word.",
        provider=provider,
        sink=sink,
        feature="provider_switch_demo",
        model=model,
    )
    return res.text.strip()

print("gemini:", run_demo(gemini, "gemini-2.5-flash"))
print("openai:", run_demo(openai, "gpt-5.4-nano"))

## 3. Cost is recorded the same way for both

Every call writes one record to the sink. Because pricing is keyed by model
name, OpenAI rows get OpenAI rates and Gemini rows get Gemini rates — with no
branching in the Gateway.

In [ ]:
import json

rows = [json.loads(l) for l in open("nb_switch_invocations.jsonl", encoding="utf-8")]
for r in rows[-2:]:
    print(
        f"{r['provider']:8} {r['model']:18} "
        f"tok {r['input_tokens']}/{r['output_tokens']:<3} "
        f"${r['estimated_cost_usd']:.8f}  {round(r['latency_ms'])}ms"
    )

## 4. Adding Claude later is the same move

To add a third vendor you write **one file** — `providers/anthropic.py` — with a
class exposing `name` and `generate(...) -> LLMResult`, and add its model rates
to `pricing.py`. No call site, Gateway, eval, PII, or drift code changes.

```python
class AnthropicProvider:
    def __init__(self, api_key):
        self.name = "anthropic"
        self._client = Anthropic(api_key=api_key)

    def generate(self, prompt, *, model):
        resp = self._client.messages.create(
            model=model,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
        )
        return LLMResult(
            text=resp.content[0].text,
            input_tokens=resp.usage.input_tokens,
            output_tokens=resp.usage.output_tokens,
        )
```

## Wrap-up

The same Gateway call ran against Gemini and OpenAI by swapping one injected
object. Vendor choice becomes a one-line decision instead of a rewrite — which
is what lets you A/B providers, fail over, or migrate without touching feature
code.